In [17]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python path being used:", sys.executable)

Python path being used: /opt/anaconda3/bin/python


In [18]:
from pyspark import SparkContext, SparkConf

In [19]:
conf = SparkConf().setAppName("RDD_LAB_2").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf = conf)
sc.setLogLevel("Error")

In [20]:
print("Spark Version:", sc.version)
print("App Name:", sc.appName)

Spark Version: 4.1.1
App Name: RDD_LAB_2


In [21]:
import warnings

In [22]:
warnings.filterwarnings('ignore')

In [23]:
numbers = [10, 25, 4, 8 , 40, 17, 53, 6, 91, 3, 44]
rdd_nums = sc.parallelize(numbers)

print("Type", type(rdd_nums))
print("Partition count:", rdd_nums.getNumPartitions())
print("Count:", rdd_nums.count())
print("First 5:", rdd_nums.take(5))

Type <class 'pyspark.core.rdd.RDD'>
Partition count: 10
Count: 11
First 5: [10, 25, 4, 8, 40]


In [24]:
employees = [
    "Ram,Engineering,72000",
    "Hari,Marketing,55000",
    "Sita,Engineering,88000",
    "Gita,HR,49000",
    "Adam,Engineering,95000",
    "Eve,Marketing,61000",
    "Harry,HR,52000",
    "Hiro,Engineering,79000"
]
    

In [25]:
rdd_emp = sc.parallelize(employees)
print("Employee Count", rdd_emp.count())
print("First record", rdd_emp.first())

Employee Count 8
First record Ram,Engineering,72000


In [26]:
rdd_range = sc.range(1,21)

In [27]:
print(rdd_range)

PythonRDD[17] at RDD at PythonRDD.scala:58


In [28]:
rdd_range.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

In [29]:
print("Range sum:", rdd_range.sum())

Range sum: 210


In [31]:
import shutil
emp_path = "lab_output1/employees"

if os.path.exists(emp_path):
    shutil.rmtree(emp_path)

In [32]:
rdd_emp.saveAsTextFile(emp_path)
print("Employees Saved to:", emp_path)

Employees Saved to: lab_output1/employees


In [33]:
files = os.listdir(emp_path)
part_files = [f for f in files if f.startswith("part")]

print("Files written:", files)
print("Number of part files:", len(part_files))

Files written: ['.part-00006.crc', '._SUCCESS.crc', '.part-00007.crc', '.part-00005.crc', '.part-00004.crc', '.part-00000.crc', '.part-00001.crc', 'part-00005', 'part-00002', '.part-00003.crc', 'part-00003', '.part-00002.crc', 'part-00004', '.part-00009.crc', '.part-00008.crc', '.ipynb_checkpoints', '_SUCCESS', 'part-00008', 'part-00001', 'part-00006', 'part-00007', 'part-00000', 'part-00009']
Number of part files: 10


In [34]:
rdd_emp_loaded = sc.textFile("lab_output1/employees")
rdd_emp_loaded.collect()

['Hari,Marketing,55000',
 'Sita,Engineering,88000',
 'Gita,HR,49000',
 'Harry,HR,52000',
 'Ram,Engineering,72000',
 'Adam,Engineering,95000',
 'Eve,Marketing,61000',
 'Hiro,Engineering,79000']

In [35]:
def parse_row(line):
    parts = line.split(",")
    name = parts[0]
    dept = parts[1]
    salary = int(parts[2])
    return(name, dept, salary)

In [37]:
rdd_parsed = rdd_emp_loaded.map(parse_row)
rdd_parsed.collect()

[('Hari', 'Marketing', 55000),
 ('Sita', 'Engineering', 88000),
 ('Gita', 'HR', 49000),
 ('Harry', 'HR', 52000),
 ('Ram', 'Engineering', 72000),
 ('Adam', 'Engineering', 95000),
 ('Eve', 'Marketing', 61000),
 ('Hiro', 'Engineering', 79000)]

In [39]:
rdd_eng = rdd_parsed.filter(lambda r: r[1] == "Engineering")
print("Engineering count:", rdd_eng.count())

Engineering count: 4


In [40]:
rdd_hr = rdd_parsed.filter(lambda r: r[1] == "HR")
print("HR Count:", rdd_hr.count())

HR Count: 2


In [43]:
rdd_eng_name = rdd_eng.map(lambda r: r[0])
print("Engineering name:",rdd_eng_name.collect())

Engineering name: ['Sita', 'Ram', 'Adam', 'Hiro']


In [44]:
rdd_salary = rdd_parsed.map(lambda r: int(r[2]) > 70000)
print("Salary greater than 70000:", rdd_salary.collect())

Salary greater than 70000: [False, True, False, False, True, True, False, True]


In [46]:
rdd_salary = rdd_parsed.filter(lambda r: r[2] > 70000)
print("Salary greater than 70000:", rdd_salary.collect())

Salary greater than 70000: [('Sita', 'Engineering', 88000), ('Ram', 'Engineering', 72000), ('Adam', 'Engineering', 95000), ('Hiro', 'Engineering', 79000)]
